In [3]:
# Task 9: Spark SQL Implementation
# Create temporary views.
# Write SQL queries for:
# 1. Top 10 infection countries
# 2. Death percentage ranking
# 3. Rolling 7-day average
# Compare physical plans with DataFrame API.

In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

In [5]:
spark = SparkSession.builder\
    .appName("Spark SQL implementation")\
.master("yarn")\
.getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/23 14:38:26 WARN Utils: Your hostname, Parashus-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.2.130 instead (on interface en0)
26/02/23 14:38:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/23 14:38:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/23 14:38:28 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [6]:
worldometer_data = spark.read.parquet("hdfs:///data/covid/staging/worldometer_data")
day_wise = spark.read.parquet("hdfs:///data/covid/staging/day_wise")

### Create temporary views.

In [7]:
worldometer_data.createOrReplaceTempView("worldometer_table")
day_wise.createOrReplaceTempView("day_wise_table")

### 1. Top 10 infection countries

In [8]:
sql_top10_infected_countries = spark.sql("""
SELECT `Country/Region` AS Country, `TotalCases`, `Population`, ROUND(TotalCases/Population * 1000,2) AS Confirmed_Per_1000
from worldometer_table
ORDER BY Confirmed_Per_1000 DESC LIMIT(10)
""")

In [9]:
sql_top10_infected_countries.show()

+-------------+----------+----------+------------------+
|      Country|TotalCases|Population|Confirmed_Per_1000|
+-------------+----------+----------+------------------+
|        Qatar|    112092|   2807805|             39.92|
|French Guiana|      8127|    299385|             27.15|
|      Bahrain|     42889|   1706669|             25.13|
|   San Marino|       699|     33938|              20.6|
|        Chile|    366671|  19132514|             19.16|
|       Panama|     71418|   4321282|             16.53|
|       Kuwait|     70045|   4276658|             16.38|
|         Oman|     80713|   5118446|             15.77|
|          USA|   5032179| 331198130|             15.19|
| Vatican City|        12|       801|             14.98|
+-------------+----------+----------+------------------+



In [10]:
sql_top10_infected_countries.write.mode("overwrite").parquet("hdfs:///data/covid/analytics/sql_top10_infected_countries")

### 2. Death percentage ranking

In [11]:
sql_death_percentage_ranking = spark.sql("""
with death_percentage_calc AS (
    SELECT `Country/Region` AS Country, 
    TotalDeaths/TotalCases * 100 AS Death_Percentage
    FROM worldometer_table
)
SELECT Country, Death_percentage, DENSE_RANK() OVER(ORDER BY Death_Percentage DESC) AS Rank
FROM death_percentage_calc
""")

In [12]:
sql_death_percentage_ranking.show()

26/02/23 14:38:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 4:>                                                          (0 + 1) / 1]

+---------------+------------------+----+
|        Country|  Death_percentage|Rank|
+---------------+------------------+----+
|          Yemen|28.733031674208142|   1|
|         France|15.494318443207433|   2|
|             UK| 15.06260263391901|   3|
|          Italy|14.119757307266337|   4|
|        Belgium|13.855083054610867|   5|
|        Hungary|13.051990428540353|   6|
|         Mexico|10.918109317253451|   7|
|    Netherlands|10.798146783194692|   8|
|   Sint Maarten|              10.0|   9|
| Western Sahara|              10.0|   9|
|           Chad| 8.067940552016985|  10|
|          Spain| 8.038811948213127|  11|
|Channel Islands| 7.872696817420436|  12|
|     Montserrat|7.6923076923076925|  13|
|         Canada| 7.562351869501775|  14|
|    Isle of Man| 7.142857142857142|  15|
|         Sweden| 7.034538289799554|  16|
|        Ireland| 6.704080084938571|  17|
|        Ecuador|6.4912687630471515|  18|
|          Sudan|6.4770797962648565|  19|
+---------------+-----------------

In [13]:
sql_death_percentage_ranking.write.mode("overwrite").parquet("hdfs:///data/covid/analytics/sql_death_percentage_ranking")

26/02/23 14:38:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

### 3. Rolling 7-day average

In [14]:
sql_7day_rolling_average = spark.sql("""
with select_columns AS (
    SELECT Date, Confirmed, Deaths, Recovered
    FROM day_wise_table
)
SELECT Date,
Confirmed, ROUND(AVG(Confirmed) OVER(ORDER BY Date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),2) AS confirmed_rolling_avg,
Deaths, ROUND(Avg(Deaths) OVER(ORDER BY Date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),2) AS deaths_rolling_avg,
Recovered, ROUND(Avg(Recovered) OVER(ORDER BY Date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW),2) AS recovered_rolling_avg
FROM select_columns
""")

In [15]:
sql_7day_rolling_average.show()

26/02/23 14:38:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+---------+---------------------+------+------------------+---------+---------------------+
|      Date|Confirmed|confirmed_rolling_avg|Deaths|deaths_rolling_avg|Recovered|recovered_rolling_avg|
+----------+---------+---------------------+------+------------------+---------+---------------------+
|2020-01-22|      555|                555.0|    17|              17.0|       28|                 28.0|
|2020-01-23|      654|                604.5|    18|              17.5|       30|                 29.0|
|2020-01-24|      941|               716.67|    26|             20.33|       36|                31.33|
|2020-01-25|     1434|                896.0|    42|             25.75|       39|                33.25|
|2020-01-26|     2118|               1140.4|    56|              31.8|       52|                 37.0|
|2020-01-27|     2927|              1438.17|    82|             40.17|       61|                 41.0|
|2020-01-28|     5578|              2029.57|   131|             53.14|   

In [16]:
sql_7day_rolling_average.write.mode("overwrite").parquet("hdfs:///data/covid/analytics/sql_7day_rolling_average")

26/02/23 14:38:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:38:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

In [22]:
#---------------------------------------------------------------------------
# 4. Compare Physical Plans with DataFrame API
#---------------------------------------------------------------------------
# Top 10 Infection Countries – DataFrame API
df_top10_infected = worldometer_data \
    .selectExpr("`Country/Region` AS Country",
                "TotalCases",
                "Population",
                "ROUND(TotalCases/Population * 1000,2) AS Confirmed_Per_1000") \
    .orderBy("Confirmed_Per_1000", ascending=False) \
    .limit(10)

print("SQL Physical Plan:")
sql_top10_infected_countries.explain()

print("\nDataFrame Physical Plan:")
df_top10_infected.explain()
print("-----------------------------------------------------------------------------")
# Death Percentage Ranking – DataFrame API
window_spec = Window.orderBy(col("Death_Percentage").desc())

df_death_percentage = worldometer_data \
    .selectExpr("`Country/Region` AS Country",
                "TotalDeaths",
                "TotalCases",
                "(TotalDeaths/TotalCases) * 100 AS Death_Percentage") \
    .withColumn("Rank", dense_rank().over(window_spec))

print("\nSQL Physical Plan:")
sql_death_percentage_ranking.explain()

print("\nDataFrame Physical Plan:")
df_death_percentage.explain()
print("-----------------------------------------------------------------------------")


# Rolling 7-Day Average – DataFrame API
window_7day = Window.orderBy("Date").rowsBetween(-6, 0)

df_rolling_avg = day_wise \
    .select("Date", "Confirmed", "Deaths", "Recovered") \
    .withColumn("confirmed_rolling_avg",
                round(avg("Confirmed").over(window_7day), 2)) \
    .withColumn("deaths_rolling_avg",
                round(avg("Deaths").over(window_7day), 2)) \
    .withColumn("recovered_rolling_avg",
                round(avg("Recovered").over(window_7day), 2))

print("\nSQL Physical Plan:")
sql_7day_rolling_average.explain()

print("\nDataFrame Physical Plan:")
df_rolling_avg.explain()

SQL Physical Plan:
== Physical Plan ==
TakeOrderedAndProject(limit=10, orderBy=[Confirmed_Per_1000#21 DESC NULLS LAST], output=[Country#20,TotalCases#3L,Population#2L,Confirmed_Per_1000#21])
+- *(1) Project [Country/Region#0 AS Country#20, TotalCases#3L, Population#2L, round(((cast(TotalCases#3L as double) / cast(Population#2L as double)) * 1000.0), 2) AS Confirmed_Per_1000#21]
   +- *(1) ColumnarToRow
      +- FileScan parquet [Country/Region#0,Population#2L,TotalCases#3L] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://localhost:9000/data/covid/staging/worldometer_data], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Country/Region:string,Population:bigint,TotalCases:bigint>



DataFrame Physical Plan:
== Physical Plan ==
TakeOrderedAndProject(limit=10, orderBy=[Confirmed_Per_1000#157 DESC NULLS LAST], output=[Country#156,TotalCases#3L,Population#2L,Confirmed_Per_1000#157])
+- *(1) Project [Country/Region#0 AS Country#156, Tot

26/02/23 14:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/02/23 14:48:21 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [ ]:
spark.stop()